# 06b | Statistical inference: running tests in Python

What R does with a single `t.test()` is spread across `scipy.stats` and `statsmodels` in Python. This page walks through the common tests and has you hand-write a few things interviewers often ask you to implement on the spot (bootstrap, permutation test, power simulation).

About 60 minutes.

| R | Python |
|---|---|
| `t.test(x, y)` (Welch by default) | `stats.ttest_ind(x, y, equal_var=False)` |
| `t.test(x, mu = 0)` | `stats.ttest_1samp(x, 0)` |
| `t.test(x, y, paired = TRUE)` | `stats.ttest_rel(x, y)` |
| `prop.test()` | `statsmodels.stats.proportion.proportions_ztest` |
| `chisq.test(table)` | `stats.chi2_contingency(table)` |
| `wilcox.test(x, y)` | `stats.mannwhitneyu(x, y)` |
| `p.adjust(p, "BH")` | `statsmodels.stats.multitest.multipletests(p, method="fdr_bh")` |

⚠️ `stats.ttest_ind` defaults to `equal_var=True` (assumes equal variances), the opposite of R's default. **Make a habit of writing `equal_var=False`.**

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(7)
a = rng.normal(10.0, 3.0, 40)       # control
b = rng.normal(11.5, 4.0, 45)       # treatment
print(a.mean().round(3), b.mean().round(3))
print(stats.ttest_ind(b, a, equal_var=False))

## 1. Computing a confidence interval by hand

95% confidence interval for a mean:

$$\bar x \pm t_{0.975,\,n-1} \cdot \frac{s}{\sqrt n}$$

Interviewers often ask you to write this "without a built-in function".

### ✏️ Exercise 1 (Fill in the blanks)

Compute the 95% confidence interval for the mean of `a`; store the lower bound in `lo` and the upper bound in `hi`.

In [ ]:
n = len(a)
se = a.std(ddof=____) / np.sqrt(n)
tcrit = stats.t.ppf(____, df=n - 1)
lo = a.mean() - tcrit * se
hi = a.mean() + tcrit * se
print(lo, hi)

<details>
<summary><b>Hint</b></summary>

The sample standard deviation uses `ddof=1`; the quantile for a two-sided 95% interval is 0.975.

</details>

<details>
<summary><b>Answer</b></summary>

```python
n = len(a)
se = a.std(ddof=1) / np.sqrt(n)
tcrit = stats.t.ppf(0.975, df=n - 1)
lo = a.mean() - tcrit * se
hi = a.mean() + tcrit * se
print(lo, hi)
```

</details>

## 2. Two-proportion z-test

The most common test in A/B experiments: do the two groups' conversion rates differ?

$$z = \frac{\hat p_2 - \hat p_1}{\sqrt{\hat p(1 - \hat p)\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}}, \qquad \hat p = \frac{x_1 + x_2}{n_1 + n_2}$$

The test computes the standard error with the **pooled proportion** $\hat p$ (because under H₀ both groups have the same proportion).

### ✏️ Exercise 2 (Write it yourself)

Write a function `two_prop_z(x1, n1, x2, n2)`: `x` is the number of conversions and `n` the number of users. Return `(z, p)`, where `p` is the two-sided p-value.

Example: control has 120 conversions out of 1000, treatment 150 out of 1000; `z` is about 1.96 and `p` about 0.0497.

In [ ]:
import numpy as np
from scipy import stats

def two_prop_z(x1, n1, x2, n2):
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

Follow the formula step by step: the two proportions → pooled proportion → standard error → z. The two-sided p-value is `2 * stats.norm.sf(abs(z))`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np
from scipy import stats

def two_prop_z(x1, n1, x2, n2):
    p1, p2 = x1 / n1, x2 / n2
    pooled = (x1 + x2) / (n1 + n2)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se
    p = 2 * stats.norm.sf(abs(z))
    return z, p
```

</details>

## 3. Bootstrap confidence intervals

A recipe that works for any statistic, especially ones without a ready-made formula (median, quantiles, ratios):

1. Draw n values from the sample **with replacement** and compute the statistic once;
2. Repeat a few thousand times;
3. Take the 2.5% and 97.5% quantiles of those values (the percentile method).

`rng.choice(x, size=len(x), replace=True)` is one resample with replacement.

### ✏️ Exercise 3 (Write it yourself)

Write a function `bootstrap_ci(x, stat, n_boot, rng)`: for data `x` and a statistic function `stat` (e.g. `np.median`), run `n_boot` bootstrap resamples and return the 95% percentile confidence interval `(lo, hi)`.

In [ ]:
import numpy as np

def bootstrap_ci(x, stat, n_boot, rng):
    # Write your code below
    pass

rng = np.random.default_rng(0)
x = rng.exponential(10.0, 200)
print(np.median(x), bootstrap_ci(x, np.median, 2000, rng))

<details>
<summary><b>Hint</b></summary>

Loop `n_boot` times: `sample = rng.choice(x, size=len(x), replace=True)` and append `stat(sample)` to a list. Finish with `np.percentile(values, [2.5, 97.5])`.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def bootstrap_ci(x, stat, n_boot, rng):
    values = []
    for _ in range(n_boot):
        sample = rng.choice(x, size=len(x), replace=True)
        values.append(stat(sample))
    lo, hi = np.percentile(values, [2.5, 97.5])
    return lo, hi

rng = np.random.default_rng(0)
x = rng.exponential(10.0, 200)
print(np.median(x), bootstrap_ci(x, np.median, 2000, rng))
```

</details>

## 4. Permutation test

Tests whether two group means differ, without any distributional assumption:

1. Compute the observed difference in means;
2. Pool the two groups, **shuffle randomly**, split back into groups of the original sizes and compute the difference in means;
3. Repeat a few thousand times to get the distribution of the difference under H₀;
4. p-value = the share of shuffled differences whose **absolute value** is ≥ the absolute observed difference.

`rng.permutation(x)` returns a shuffled copy.

### ✏️ Exercise 4 (Write it yourself)

Write a function `perm_test(a, b, n_perm, rng)` that returns the two-sided p-value.

In [ ]:
import numpy as np

def perm_test(a, b, n_perm, rng):
    observed = b.mean() - a.mean()
    pooled = np.concatenate([a, b])
    # Write your code below
    pass

rng = np.random.default_rng(7)
a = rng.normal(10.0, 3.0, 40)
b = rng.normal(11.5, 4.0, 45)
print(perm_test(a, b, 5000, np.random.default_rng(1)))

<details>
<summary><b>Hint</b></summary>

Inside the loop: `shuffled = rng.permutation(pooled)`; treat the first `len(a)` values as group a and the rest as group b, and compute the difference in means; if `abs(diff) >= abs(observed)`, increment a counter. Return `count / n_perm` at the end.

</details>

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np

def perm_test(a, b, n_perm, rng):
    observed = b.mean() - a.mean()
    pooled = np.concatenate([a, b])
    count = 0
    for _ in range(n_perm):
        shuffled = rng.permutation(pooled)
        diff = shuffled[len(a):].mean() - shuffled[:len(a)].mean()
        if abs(diff) >= abs(observed):
            count += 1
    return count / n_perm

rng = np.random.default_rng(7)
a = rng.normal(10.0, 3.0, 40)
b = rng.normal(11.5, 4.0, 45)
print(perm_test(a, b, 5000, np.random.default_rng(1)))
```

</details>

## 5. Estimating power by simulation

"With 50 people per group and a true effect of 0.5 standard deviations, what is the power?"

You can answer without a formula: **simulate many experiments under the assumed truth and see what share of them come out significant.** This idea handles any design, however complicated, and mentioning it in an interview earns points.

Trick: generate a matrix in one go (each row is one experiment); `stats.ttest_ind(..., axis=1)` tests each row separately, so no loop is needed.

### ✏️ Exercise 5 (Fill in the blanks)

Estimate the power with `n = 50` per group, control N(0, 1), treatment N(0.5, 1), and two-sided α = 0.05; store it in `power`.

In [ ]:
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
n, n_sim = 50, 5000
control = rng.normal(0.0, 1.0, size=(n_sim, n))     # each row is one experiment
treat = rng.normal(____, 1.0, size=(n_sim, n))
pvals = stats.ttest_ind(treat, control, axis=1, equal_var=False).pvalue
power = (pvals < ____).mean()
print(power)

<details>
<summary><b>Answer</b></summary>

```python
import numpy as np
from scipy import stats

rng = np.random.default_rng(0)
n, n_sim = 50, 5000
control = rng.normal(0.0, 1.0, size=(n_sim, n))     # each row is one experiment
treat = rng.normal(0.5, 1.0, size=(n_sim, n))
pvals = stats.ttest_ind(treat, control, axis=1, equal_var=False).pvalue
power = (pvals < 0.05).mean()
print(power)
```

The theoretical value is about 0.70. Change the treatment mean to 0 (no true effect) and rerun: what you get is the Type I error rate, which should be close to 0.05. This is the standard way to check whether a testing procedure is trustworthy.

</details>

## 6. Multiple-testing correction

In [ ]:
from statsmodels.stats.multitest import multipletests

pvals = [0.001, 0.008, 0.012, 0.03, 0.04, 0.2, 0.5, 0.7, 0.9, 0.95]
reject_bonf = multipletests(pvals, alpha=0.05, method="bonferroni")[0]
reject_bh = multipletests(pvals, alpha=0.05, method="fdr_bh")[0]
print(reject_bonf.sum(), reject_bh.sum())

The next exercise numbers the sorted p-values 1, 2, 3… (the i in the formula starts at 1). `enumerate` counts from 0 by default; add `start=1` to count from 1:

In [ ]:
for i, p in enumerate(sorted([0.3, 0.01, 0.2]), start=1):
    print(i, p)

### ✏️ Exercise 6 (Write it yourself)

Implement **Benjamini–Hochberg** yourself, without `multipletests`: write a function `bh_reject_count(pvals, alpha)` that returns the number of rejected hypotheses.

Rule: sort the p-values in increasing order, $p_{(1)} \le \dots \le p_{(m)}$. Find the **largest** i with $p_{(i)} \le \frac{i}{m}\alpha$ and reject the first i. If none qualifies, return 0.

In [ ]:
def bh_reject_count(pvals, alpha):
    # Write your code below
    pass

<details>
<summary><b>Hint</b></summary>

`for i, p in enumerate(sorted(pvals), start=1):` if `p <= i / m * alpha`, set `k` to `i` (don't break: you want the largest i).

</details>

<details>
<summary><b>Answer</b></summary>

```python
def bh_reject_count(pvals, alpha):
    m = len(pvals)
    k = 0
    for i, p in enumerate(sorted(pvals), start=1):
        if p <= i / m * alpha:
            k = i
    return k
```

Note that it is the "largest i": after sorting `[0.04, 0.03, 0.02, 0.01]`, the 1st value 0.01 ≤ 0.0125 and the 4th value 0.04 ≤ 0.05, so all four are rejected, even if some p-value in between fails its own threshold on its own.

</details>

## Summary

- `stats.ttest_ind(..., equal_var=False)`; use a z-test for proportions;
- Confidence interval: estimate ± critical value × SE;
- Bootstrap: resample with replacement → statistic → quantiles;
- Permutation test: shuffle labels → null distribution → p-value;
- Both power and the Type I error rate can be obtained by simulation.

On to **07a**.